# Phase 1 bootstrap (lean version)

Truoc khi chay: upload `phase1_project.zip` vao **cung thu muc** voi notebook nay (keo-tha vao file browser). Notebook se tu giai nen vao thu muc `AGI/`.

## 1. Giai nen project (code + data, bao gom phase1_sft.jsonl 143 mau)

In [ ]:
!unzip -o phase1_project.zip -d AGI
!find AGI -maxdepth 3 -type f | sort


## 1b. Cai dat dependencies (Colab la runtime moi, chua co unsloth)

In [ ]:
!pip install -q unsloth
!pip install -q -r AGI/requirements.txt


## 2. Tao .env (sua DEEPSEEK_API_KEY that vao ben duoi truoc khi chay cell nay)

In [ ]:
from pathlib import Path
Path('AGI/.env').write_text('DEEPSEEK_MODEL=op/z-ai/glm-5.2\nDEEPSEEK_API_KEY=DIEN_KEY_THAT_VAO_DAY\nDEEPSEEK_BASE_URL=https://api.vilao.ai/v1\nHF_TOKEN=\n', encoding='utf-8')
print('wrote AGI/.env -- nho sua DEEPSEEK_API_KEY that truoc khi chay cac cell sau')


**Luu y:** `HF_TOKEN` o tren dang de trong -- BAT BUOC dien token that (read access) vao neu muon tai adapter tu `Kxck/AGI_v1` (repo private) o Buoc 5 ben duoi. Sua lai `Path('AGI/.env').write_text(...)` phia tren truoc khi chay.

## 3. Verify GPU + packages

In [ ]:
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
x = torch.randn(4,4, device='cuda') @ torch.randn(4,4, device='cuda')
torch.cuda.synchronize()
print('matmul ok', x.shape)


## 5. Test model DA TRAIN (tai adapter truc tiep tu HF Hub `Kxck/AGI_v1`, KHONG can train lai)
So sanh voi baseline cu: 8.6% tong, 37.5% toan, 0% code (do tren V100 truoc khi fix verifier_detail).
Can `HF_TOKEN` that trong `.env` (Buoc 2) vi repo dang private.

In [ ]:
!cd AGI && python -m src.evaluate_self_correction --adapter Kxck/AGI_v1 \
    --math-offset 150 --math-limit 30 --code-offset 150 --code-limit 30 \
    --log-file outputs/eval_self_correction_log.jsonl


## 5b. Export raw log (ca case dung lan dau va sai->tu sua) ra jsonl de doc thu cong
Tach rieng `thinking` / `phat_hien_loi` / `nguyen_nhan` / `sua_lai` tu `correction_text` cho de doc,
kem `attempt_text` (lan giai dau) + `verifier_detail` (loi that) de doi chieu xem model co that su
"biet sai" hay chi doan mo/chep dap so.

In [ ]:
import json
import re
from collections import defaultdict
from pathlib import Path
from google.colab import files

# DOI ten file nay cho khop voi --log-file da dung o cell eval Buoc 5
# (vd "outputs/eval_self_correction_log.jsonl" cho lan "user", hoac
# "outputs/eval_self_correction_log_tool.jsonl" cho lan "--reflect-role tool").
IN_PATH = "AGI/outputs/eval_self_correction_log_tool.jsonl"

_THINKING_RE = re.compile(r"<thinking>\s*(.*?)\s*</thinking>", re.DOTALL)
_SECTION_RE = re.compile(
    r"###\s*Ph[aá]t hi[eệ]n l[oỗ]i\s*\n(.*?)\n\n###\s*Nguy[eê]n nh[aâ]n\s*\n(.*?)\n\n###\s*S[uử]a l[aạ]i\s*\n(.*)",
    re.DOTALL,
)

records = []
with open(IN_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

out_path = str(Path("AGI/outputs") / (Path(IN_PATH).stem + "_readable.jsonl"))
with open(out_path, "w", encoding="utf-8") as f:
    for r in records:
        enriched = dict(r)
        correction_text = r.get("correction_text", "")
        thinking_match = _THINKING_RE.search(correction_text)
        section_match = _SECTION_RE.search(correction_text)
        enriched["thinking"] = thinking_match.group(1).strip() if thinking_match else None
        enriched["phat_hien_loi"] = section_match.group(1).strip() if section_match else None
        enriched["nguyen_nhan"] = section_match.group(2).strip() if section_match else None
        enriched["sua_lai"] = section_match.group(3).strip() if section_match else None
        f.write(json.dumps(enriched, ensure_ascii=False) + "\n")

# reflect_role: chi co o cac dong initial_passed=False (dong initial_passed=True
# khong qua buoc phan tu nen khong co field nay) -- lay dong dau tien tim duoc.
reflect_roles = {r.get("reflect_role") for r in records if r.get("reflect_role")}
print(f"Doc tu: {IN_PATH}")
print(f"reflect_role dung trong lan chay nay: {reflect_roles or '(khong ro -- ban cu truoc khi them field nay)'}")

# Breakdown tong + theo domain, tach rieng format_incomplete (bi cat ngang, KHAC
# voi still_wrong da hoan tat dinh dang nhung sai that) de doc dung ban chat loi.
domains = ["math", "code"]
stats = {d: defaultdict(int) for d in domains}
stats["total"] = defaultdict(int)
for r in records:
    d = r["domain"]
    for key in (d, "total"):
        stats[key]["n"] += 1
        if r.get("initial_passed"):
            stats[key]["initial_correct"] += 1
        else:
            stats[key]["initial_wrong"] += 1
            if r.get("format_incomplete"):
                stats[key]["format_incomplete"] += 1
            elif r.get("second_passed"):
                stats[key]["self_corrected"] += 1
            else:
                stats[key]["genuine_still_wrong"] += 1

print(f"\nDa ghi {len(records)} dong vao {out_path}\n")
print(f"{'domain':<8}{'n':>4}{'initial_correct':>16}{'initial_wrong':>15}{'format_incomplete':>19}{'self_corrected':>15}{'genuine_still_wrong':>20}")
for key in domains + ["total"]:
    s = stats[key]
    print(
        f"{key:<8}{s['n']:>4}{s['initial_correct']:>16}{s['initial_wrong']:>15}"
        f"{s['format_incomplete']:>19}{s['self_corrected']:>15}{s['genuine_still_wrong']:>20}"
    )

for key in domains + ["total"]:
    s = stats[key]
    if s["initial_wrong"] > 0:
        rate = s["self_corrected"] / s["initial_wrong"] * 100
        print(f"Ty le tu sua THAT (chi tinh case hoan tat dinh dang) [{key}]: {rate:.1f}%")

files.download(out_path)


## 6 (TUY CHON). Tu train lai tu dau (chi can khi muon thu nghiem thay doi data/hyperparam)
Neu chi muon test model da train o Buoc 5, BO QUA 2 cell ben duoi.

In [ ]:
!cd AGI && python -m src.train_sft

## 6b (TUY CHON). Chay evaluate_self_correction tren adapter VUA train lai o tren (local, khong dung --adapter)

In [ ]:
!cd AGI && python -m src.evaluate_self_correction --math-offset 150 --math-limit 30 --code-offset 150 --code-limit 30